In [1]:
import os
import pandas as pd
!pip install streamlit
import streamlit as st
import plotly.express as px


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 54.5 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 91.6 MB/s eta 0:00:00:00:010:01


In [3]:
st.set_page_config(page_title="AI Job Risk Report Generator", page_icon=" ", layout="wide")
API_KEY = os.getenv("OPENAI_API_KEY") 
client = OpenAI(api_key=API_KEY) if API_KEY else None

2026-05-27 07:09:08.184 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.


In [4]:
# -----------------------------# Data loading# -----------------------------
@st.cache_data
def load_data(path):
    return pd.read_csv(path)
    
@st.cache_data
def convert_df_to_csv(df):
    return df.to_csv(index=False).encode("utf-8")
df = load_data("https://raw.githubusercontent.com/rgsvm/AI-Automation/main/your_data_file.csv")


2026-05-27 07:09:13.573 WARNING streamlit.runtime.caching.cache_data_api: No runtime found, using MemoryCacheStorageManager
2026-05-27 07:09:13.576 WARNING streamlit.runtime.caching.cache_data_api: No runtime found, using MemoryCacheStorageManager
2026-05-27 07:09:13.577 WARNING streamlit.runtime.caching.cache_data_api: No runtime found, using MemoryCacheStorageManager
2026-05-27 07:09:13.578 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.


In [5]:
df.head()

,Year,Country,AI_Investment_BillionUSD,Automation_Rate_Percent,Employment_Rate_Percent,Average_Salary_USD,Productivity_Index,Reskilling_Investment_MillionUSD,AI_Policy_Index,Job_Displacement_Million,Job_Creation_Million,AI_Readiness_Score
0,2015,United States,52.46,10.75,65.50,38392.64,65.78,709.03,0.64,0.19,0.14,47.87
1,2016,United States,60.64,11.64,65.25,39371.74,67.02,815.38,0.75,0.16,0.85,43.85
2,2017,United States,66.11,12.46,64.93,40772.29,68.51,921.74,0.51,0.17,1.23,37.13
3,2018,United States,72.45,13.35,64.84,43974.92,71.09,1028.09,0.69,0.27,0.76,42.28
4,2019,United States,79.11,14.65,64.55,43745.39,69.89,1134.45,0.57,0.25,0.85,43.95


In [13]:

#-----------------------------# Helpers# -----------------------------
def safe_unique_list(df, col):
    if col in df.columns:
        return sorted(df[col].dropna().astype(str).unique().tolist())
    return []
    
def build_job_prompt(row):
    return f"""You are an AI workforce analyst.

Analyze this job role for AI impact by 2030:

Job title: {row.get('job_title', 'N/A')}
Industry: {row.get('industry', 'N/A')}
Category: {row.get('job_category', 'N/A')}
Risk score: {row.get('risk_score', 'N/A')}
Risk level: {row.get('risk_level', 'N/A')}
Main skills: {row.get('skills', 'N/A')}
Return the response in this exact format:

Summary:Why this role is at risk:
Skills that reduce risk:
Recommended action plan:
Executive note:
Tags:""".strip()

In [20]:
def make_fallback_report(row, message):
    return {
        "job_title": row.get("job_title", "N/A"),
        "industry": row.get("industry", "N/A"),
        "risk_score": row.get("risk_score", "N/A"),
        "risk_level": row.get("risk_level", "N/A"),
        "generated_report": message
    }

In [22]:
def generate_job_report(row, model="gpt-4o-mini"):
    if client is None:
        return make_fallback_report(row, "OPENAI_API_KEY is missing.")

    try:
        prompt = build_job_prompt(row)
        response = client.responses.create(model=model, input=prompt)
        text = getattr(response, "output_text", "").strip()

        if not text:
            return make_fallback_report(row, "No output returned from the model.")

        return {
            "job_title": row.get("job_title", "N/A"),
            "industry": row.get("industry", "N/A"),
            "risk_score": row.get("risk_score", "N/A"),
            "risk_level": row.get("risk_level", "N/A"),
            "generated_report": text
        }

    except RateLimitError:
        return make_fallback_report(row, "Rate limit reached. Please try again later.")
    except AuthenticationError:
        return make_fallback_report(row, "Authentication error. Check your API key.")
    except APIConnectionError:
        return make_fallback_report(row, "Connection error. Please check your internet or API status.")
    except BadRequestError as e:
        return make_fallback_report(row, f"Bad request error: {e}")
    except Exception as e:
        return make_fallback_report(row, f"Unexpected error: {e}")

In [26]:
def generate_reports_for_dataframe(df_input):
    reports = []
    for _, row in df_input.iterrows():
        reports.append(generate_job_report(row))
        return pd.DataFrame(reports)

In [34]:
# -----------------------------
# Sidebar filters
# -----------------------------
st.sidebar.header("Filters")
job_category = st.sidebar.selectbox("Job Category", ["All"] + safe_unique_list(df, "job_category"))
industry = st.sidebar.selectbox("Industry", ["All"] + safe_unique_list(df, "industry"))
risk_level = st.sidebar.selectbox("Risk Level", ["All"] + safe_unique_list(df, "risk_level"))
selected_job = st.sidebar.selectbox("Select Job Title", ["All"] + safe_unique_list(df, "job_title"))
show_high_risk_only = st.sidebar.checkbox("Show only high-risk jobs", value=False)

2026-05-27 07:16:30.838 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-05-27 07:16:30.839 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-05-27 07:16:30.840 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-05-27 07:16:30.841 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-05-27 07:16:30.842 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-05-27 07:16:30.844 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-05-27 07:16:30.844 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-05-27 07:16:30.845 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bar

In [30]:
# -----------------------------# Filtering# -----------------------------
filtered = df.copy()
if job_category != "All" and "job_category" in filtered.columns:filtered = filtered[filtered["job_category"].astype(str) == job_category]
if industry != "All" and "industry" in filtered.columns:filtered = filtered[filtered["industry"].astype(str) == industry]
if risk_level != "All" and "risk_level" in filtered.columns:filtered = filtered[filtered["risk_level"].astype(str) == risk_level]
if selected_job != "All" and "job_title" in filtered.columns:filtered = filtered[filtered["job_title"].astype(str) == selected_job]
if show_high_risk_only and "risk_level" in filtered.columns:filtered = filtered[filtered["risk_level"].astype(str) == "High"]

In [31]:
# -----------------------------
# Tabs
# -----------------------------
tab1, tab2, tab3 = st.tabs(["Dashboard", "Job Report", "Data Table"])

with tab1:
    left, right = st.columns(2)
    
    with left:
        if len(filtered) > 0 and "risk_score" in filtered.columns and "job_title" in filtered.columns:
            plot_df = filtered.copy()
            plot_df["risk_score"] = pd.to_numeric(plot_df["risk_score"], errors="coerce")
            plot_df = plot_df.dropna(subset=["risk_score"]).sort_values("risk_score", ascending=False).head(10)
            
            if len(plot_df) > 0:
                fig1 = px.bar(
                    plot_df,
                    x="job_title",
                    y="risk_score",
                    color="risk_level" if "risk_level" in plot_df.columns else None,
                    title="Top 10 Highest Risk Jobs"
                )
                st.plotly_chart(fig1, use_container_width=True)
        else:
            st.info("No valid risk scores available for charting.")
    
    with right:
        if len(filtered) > 0 and "risk_level" in filtered.columns:
            fig2 = px.histogram(
                filtered,
                x="risk_level",
                color="risk_level",
                title="Risk Level Distribution"
            )
            st.plotly_chart(fig2, use_container_width=True)
    
    bottom_left, bottom_right = st.columns(2)
    
    with bottom_left:
        if len(filtered) > 0 and "industry" in filtered.columns and "risk_score" in filtered.columns:
            plot_df = filtered.copy()
            plot_df["risk_score"] = pd.to_numeric(plot_df["risk_score"], errors="coerce")
            plot_df = plot_df.dropna(subset=["risk_score"])
            
            if len(plot_df) > 0:
                fig3 = px.box(
                    plot_df,
                    x="industry",
                    y="risk_score",
                    title="Risk Score by Industry"
                )
                st.plotly_chart(fig3, use_container_width=True)
    
    with bottom_right:
        if len(filtered) > 0 and "skill_category" in filtered.columns:
            fig4 = px.pie(
                filtered,
                names="skill_category",
                title="Skill Category Share"
            )
            st.plotly_chart(fig4, use_container_width=True)
    
    st.subheader("AI Insights")
    st.write("This dashboard helps identify which roles are more exposed to AI automation and what skills may improve resilience.")

with tab2:
    st.subheader("AI-Generated Job Report")
    
    if len(filtered) > 0:
        job_row = filtered.iloc[0]
        report = generate_job_report(job_row)
        
        st.write(f"**Job Title:** {report['job_title']}")
        st.write(f"**Industry:** {report['industry']}")
        st.write(f"**Risk Score:** {report['risk_score']}")
        st.write(f"**Risk Level:** {report['risk_level']}")
        
        st.markdown("### Generated Report")
        st.write(report["generated_report"])
        
        st.markdown("### Export This Report")
        single_report_df = pd.DataFrame([report])
        single_csv = convert_df_to_csv(single_report_df)
        
        st.download_button(
            label="Download selected job report as CSV",
            data=single_csv,
            file_name="selected_job_report.csv",
            mime="text/csv"
        )
    else:
        st.info("No job selected or no records available.")

with tab3:
    st.subheader("Filtered Job Data")
    st.dataframe(filtered, use_container_width=True)
    
    filtered_csv = convert_df_to_csv(filtered)
    st.download_button(
        label="Download filtered data as CSV",
        data=filtered_csv,
        file_name="filtered_ai_jobs.csv",
        mime="text/csv"
    )
    
    if st.button("Generate Reports for All Filtered Jobs"):
        with st.spinner("Generating reports..."):
            report_df = generate_reports_for_dataframe(filtered)
        
        st.success("Reports generated successfully.")
        st.dataframe(report_df, use_container_width=True)
        
        report_csv = convert_df_to_csv(report_df)
        st.download_button(
            label="Download all generated reports as CSV",
            data=report_csv,
            file_name="ai_job_reports.csv",
            mime="text/csv"
        )

2026-05-27 07:14:36.089 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-05-27 07:14:36.090 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-05-27 07:14:36.091 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-05-27 07:14:36.092 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-05-27 07:14:36.094 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-05-27 07:14:36.095 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-05-27 07:14:36.096 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-05-27 07:14:36.096 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bar